# Public History Verifier for Avalon Deception Dataset

This notebook verifies the integrity of the `public_history` column across 250 games.

**Validation Rules:**
- Hard Violations: Impossible game states (data corruption/generation errors)
- Soft Violations: Unusual but potentially valid patterns (manual review recommended)

In [1]:
import pandas as pd
import json
import re
from collections import defaultdict

In [2]:
# Load dataset
df = pd.read_csv('Deception-Dataset.csv')
print(f"Total rows: {len(df)}")
print(f"Unique games: {df['game_id'].nunique()}")
print(f"Columns: {df.columns.tolist()}")

Total rows: 1000
Unique games: 250
Columns: ['game_id', 'round_id', 'role_id', 'llm_alignment', 'player_roles', 'public_history', 'prior_summary_gold', 'discussion_log', 'matrix_tactic_scale', 'reasoning_gold', 'Overall_with_formula', 'Overall']


In [3]:
# Expected team sizes by round
TEAM_SIZES = {1: 2, 2: 3, 3: 2, 4: 3, 5: 3}

def get_next_leader(current_leader):
    """Get next leader in rotation: P1→P2→P3→P4→P5→P1"""
    player_num = int(current_leader[1])
    next_num = (player_num % 5) + 1
    return f"P{next_num}"

def parse_public_history(pub_hist):
    """
    Parse public_history string into structured data.
    
    Returns dict with keys: round, leader, team, votes, outcome
    Returns None if parsing fails
    """
    if pd.isna(pub_hist) or not isinstance(pub_hist, str):
        return None
    
    try:
        parsed = {}
        
        # Extract round number
        round_match = re.search(r'Round:\s*(\d+)', pub_hist)
        if round_match:
            parsed['round'] = int(round_match.group(1))
        else:
            return None
        
        # Extract leader
        leader_match = re.search(r'Leader:\s*(P\d)', pub_hist)
        if leader_match:
            parsed['leader'] = leader_match.group(1)
        else:
            return None
        
        # Extract team (already sorted alphabetically)
        team_match = re.search(r'Team:\s*([P\d,\s]+)', pub_hist)
        if team_match:
            team_str = team_match.group(1).strip()
            parsed['team'] = [p.strip() for p in team_str.split(',')]
        else:
            return None
        
        # Extract votes
        votes_match = re.search(r'Votes:\s*(.+?)(?:\n|Quest)', pub_hist)
        if votes_match:
            votes_str = votes_match.group(1).strip()
            votes = {}
            for vote in votes_str.split():
                if ':' in vote:
                    player, decision = vote.split(':')
                    votes[player] = decision
            parsed['votes'] = votes
        else:
            return None
        
        # Extract outcome
        outcome_match = re.search(r'Quest\s*\d+\s*Outcome:\s*(PASS|FAIL)', pub_hist)
        if outcome_match:
            parsed['outcome'] = outcome_match.group(1)
        else:
            return None
        
        return parsed
        
    except Exception as e:
        return None

In [4]:
def verify_game(game_df):
    """
    Verify a single game's public_history across all rounds.
    
    Returns tuple: (hard_violations, soft_violations)
    """
    hard_violations = []
    soft_violations = []
    
    # Sort by round_id
    game_df = game_df.sort_values('round_id').reset_index(drop=True)
    game_id = game_df['game_id'].iloc[0]
    
    # Get unique rounds (same public_history across role_ids)
    rounds_data = []
    for round_id in sorted(game_df['round_id'].unique()):
        round_rows = game_df[game_df['round_id'] == round_id]
        pub_hist = round_rows['public_history'].iloc[0]
        player_roles_str = round_rows['player_roles'].iloc[0]
        
        parsed = parse_public_history(pub_hist)
        if parsed is None:
            hard_violations.append(f"Round {round_id}: Failed to parse public_history")
            continue
        
        parsed['round_id'] = round_id
        parsed['player_roles'] = json.loads(player_roles_str) if isinstance(player_roles_str, str) else player_roles_str
        rounds_data.append(parsed)
    
    if not rounds_data:
        hard_violations.append("No valid rounds found")
        return hard_violations, soft_violations
    
    # --- HARD VIOLATIONS ---
    
    # 1. Verify game length (3-5 rounds)
    num_rounds = len(rounds_data)
    if num_rounds < 3 or num_rounds > 5:
        hard_violations.append(f"Invalid game length: {num_rounds} rounds (expected 3-5)")
    
    # 2. Verify quest numbers match round_id
    for rd in rounds_data:
        if rd['round'] != rd['round_id']:
            hard_violations.append(f"Round {rd['round_id']}: Quest number mismatch (public_history says Round {rd['round']}, but round_id is {rd['round_id']})")
    
    # 3. Verify leader rotation
    for i in range(1, len(rounds_data)):
        prev_leader = rounds_data[i-1]['leader']
        expected_leader = get_next_leader(prev_leader)
        actual_leader = rounds_data[i]['leader']
        
        if actual_leader != expected_leader:
            hard_violations.append(f"Round {rounds_data[i]['round_id']}: Leader rotation error (expected {expected_leader} after {prev_leader}, got {actual_leader})")
    
    # 4. Verify team sizes
    for rd in rounds_data:
        expected_size = TEAM_SIZES.get(rd['round_id'])
        actual_size = len(rd['team'])
        
        if expected_size and actual_size != expected_size:
            hard_violations.append(f"Round {rd['round_id']}: Team size error (expected {expected_size}, got {actual_size})")
    
    # 5. Verify leader is in team
    for rd in rounds_data:
        if rd['leader'] not in rd['team']:
            hard_violations.append(f"Round {rd['round_id']}: Leader {rd['leader']} not in team {rd['team']}")
    
    # 6. Verify valid player IDs
    valid_players = {'P1', 'P2', 'P3', 'P4', 'P5'}
    for rd in rounds_data:
        invalid_players = [p for p in rd['team'] if p not in valid_players]
        if invalid_players:
            hard_violations.append(f"Round {rd['round_id']}: Invalid player IDs in team: {invalid_players}")
        
        if rd['leader'] not in valid_players:
            hard_violations.append(f"Round {rd['round_id']}: Invalid leader ID: {rd['leader']}")
    
    # 7. Verify game termination (exactly 3 PASS or 3 FAIL)
    outcomes = [rd['outcome'] for rd in rounds_data]
    pass_count = outcomes.count('PASS')
    fail_count = outcomes.count('FAIL')
    
    if pass_count != 3 and fail_count != 3:
        hard_violations.append(f"Wrong termination: {pass_count} PASS, {fail_count} FAIL (need exactly 3 of one type)")
    
    # 8. Verify votes format (all 5 players)
    for rd in rounds_data:
        if len(rd['votes']) != 5:
            hard_violations.append(f"Round {rd['round_id']}: Invalid vote count (expected 5, got {len(rd['votes'])})")
        
        for player in valid_players:
            if player not in rd['votes']:
                hard_violations.append(f"Round {rd['round_id']}: Missing vote for {player}")
    
    # --- SOFT VIOLATIONS ---
    
    # 1. Check if both failed players reappear together in next round
    for i in range(len(rounds_data) - 1):
        curr_round = rounds_data[i]
        next_round = rounds_data[i + 1]
        
        if curr_round['outcome'] == 'FAIL':
            failed_team = set(curr_round['team'])
            next_team = set(next_round['team'])
            
            # Check if both (or all) failed players are in next team
            failed_in_next = failed_team.intersection(next_team)
            
            # Only flag if it's not round 5 (desperation) and significant overlap
            if next_round['round_id'] != 5 and len(failed_in_next) >= 2 and len(failed_team) == 2:
                soft_violations.append(f"Round {curr_round['round_id']}→{next_round['round_id']}: Both failed players {sorted(failed_team)} reappear in next team {sorted(next_team)}")
    
    # 2. Check for identical team composition in consecutive rounds
    for i in range(len(rounds_data) - 1):
        curr_team = set(rounds_data[i]['team'])
        next_team = set(rounds_data[i + 1]['team'])
        
        if curr_team == next_team:
            soft_violations.append(f"Round {rounds_data[i]['round_id']}→{rounds_data[i+1]['round_id']}: Identical team {sorted(curr_team)} despite outcome {rounds_data[i]['outcome']}")
    
    # 3. Check for all-good or all-evil teams (unusual strategy)
    for rd in rounds_data:
        player_roles = rd['player_roles']
        team_roles = [player_roles[p] for p in rd['team'] if p in player_roles]
        
        if team_roles and all(role == 'Good' for role in team_roles):
            soft_violations.append(f"Round {rd['round_id']}: All-Good team {sorted(rd['team'])}")
        elif team_roles and all(role == 'Evil' for role in team_roles):
            soft_violations.append(f"Round {rd['round_id']}: All-Evil team {sorted(rd['team'])}")
    
    return hard_violations, soft_violations

In [5]:
# Run verification on all games
print("Running verification...")

hard_violations_dict = {}
soft_violations_dict = {}

for game_id in df['game_id'].unique():
    game_df = df[df['game_id'] == game_id]
    hard_viols, soft_viols = verify_game(game_df)
    
    if hard_viols:
        hard_violations_dict[game_id] = hard_viols
    if soft_viols:
        soft_violations_dict[game_id] = soft_viols

print(f"\n{'='*70}")
print(f"VERIFICATION COMPLETE")
print(f"{'='*70}")
print(f"Total games analyzed: {df['game_id'].nunique()}")
print(f"Games with HARD violations: {len(hard_violations_dict)}")
print(f"Games with SOFT violations: {len(soft_violations_dict)}")
print(f"{'='*70}\n")

Running verification...

VERIFICATION COMPLETE
Total games analyzed: 250
Games with HARD violations: 0
Games with SOFT violations: 123



In [6]:
# Display Hard Violations
if hard_violations_dict:
    print(f"\n{'='*70}")
    print(f"HARD VIOLATIONS ({len(hard_violations_dict)} games)")
    print(f"These indicate data corruption or generation errors")
    print(f"{'='*70}\n")
    
    for game_id in sorted(hard_violations_dict.keys()):
        print(f"\n{game_id}:")
        for violation in hard_violations_dict[game_id]:
            print(f"  - {violation}")
else:
    print("\n✓ No hard violations found!")


✓ No hard violations found!


In [7]:
# Display Soft Violations (first 20 for readability)
if soft_violations_dict:
    print(f"\n{'='*70}")
    print(f"SOFT VIOLATIONS ({len(soft_violations_dict)} games)")
    print(f"Review recommended - unusual but potentially valid patterns")
    print(f"{'='*70}\n")
    
    displayed = 0
    for game_id in sorted(soft_violations_dict.keys()):
        if displayed >= 20:
            print(f"\n... and {len(soft_violations_dict) - 20} more games with soft violations")
            break
        
        print(f"\n{game_id}:")
        for violation in soft_violations_dict[game_id]:
            print(f"  - {violation}")
        displayed += 1
else:
    print("\n✓ No soft violations found!")


SOFT VIOLATIONS (123 games)
Review recommended - unusual but potentially valid patterns


G022:
  - Round 3: All-Good team ['P1', 'P3']

G023:
  - Round 1: All-Good team ['P1', 'P2']

G025:
  - Round 3: All-Good team ['P1', 'P3']

G027:
  - Round 2: All-Good team ['P1', 'P2', 'P3']

G029:
  - Round 3: All-Good team ['P1', 'P3']

G030:
  - Round 1: All-Good team ['P1', 'P2']
  - Round 2: All-Good team ['P1', 'P2', 'P3']

G032:
  - Round 4→5: Identical team ['P1', 'P4', 'P5'] despite outcome PASS

G033:
  - Round 2: All-Good team ['P1', 'P2', 'P3']
  - Round 4: All-Good team ['P1', 'P2', 'P3']

G034:
  - Round 2: All-Good team ['P1', 'P2', 'P3']

G035:
  - Round 1: All-Good team ['P1', 'P3']
  - Round 3: All-Good team ['P3', 'P5']

G036:
  - Round 4→5: Identical team ['P1', 'P2', 'P3'] despite outcome PASS

G038:
  - Round 4→5: Identical team ['P1', 'P4', 'P5'] despite outcome PASS

G039:
  - Round 4→5: Identical team ['P2', 'P3', 'P5'] despite outcome PASS

G040:
  - Round 1: All-Good 

In [8]:
# Summary statistics
print(f"\n{'='*70}")
print(f"SUMMARY STATISTICS")
print(f"{'='*70}")

# Count violation types
hard_violation_types = defaultdict(int)
for violations in hard_violations_dict.values():
    for v in violations:
        # Extract violation type (first part before colon)
        vtype = v.split(':')[0] if ':' in v else v
        hard_violation_types[vtype] += 1

soft_violation_types = defaultdict(int)
for violations in soft_violations_dict.values():
    for v in violations:
        if 'failed players' in v.lower():
            soft_violation_types['Failed players reappear'] += 1
        elif 'identical team' in v.lower():
            soft_violation_types['Identical team composition'] += 1
        elif 'all-good' in v.lower():
            soft_violation_types['All-Good team'] += 1
        elif 'all-evil' in v.lower():
            soft_violation_types['All-Evil team'] += 1

print("\nHard Violation Types:")
if hard_violation_types:
    for vtype, count in sorted(hard_violation_types.items()):
        print(f"  {vtype}: {count}")
else:
    print("  None")

print("\nSoft Violation Types:")
if soft_violation_types:
    for vtype, count in sorted(soft_violation_types.items()):
        print(f"  {vtype}: {count}")
else:
    print("  None")


SUMMARY STATISTICS

Hard Violation Types:
  None

Soft Violation Types:
  All-Evil team: 30
  All-Good team: 90
  Identical team composition: 39


In [9]:
# Export violations to CSV for detailed review
if hard_violations_dict or soft_violations_dict:
    violation_records = []
    
    for game_id, violations in hard_violations_dict.items():
        for v in violations:
            violation_records.append({
                'game_id': game_id,
                'violation_type': 'HARD',
                'violation': v
            })
    
    for game_id, violations in soft_violations_dict.items():
        for v in violations:
            violation_records.append({
                'game_id': game_id,
                'violation_type': 'SOFT',
                'violation': v
            })
    
    violations_df = pd.DataFrame(violation_records)
    violations_df.to_csv('ph-verification-results.csv', index=False)
    print(f"\n✓ Violations exported to: ph-verification-results.csv")
    print(f"  Total violation records: {len(violations_df)}")


✓ Violations exported to: ph-verification-results.csv
  Total violation records: 159
